In [9]:
from pathlib import Path
import re
from collections import Counter, defaultdict
from bs4 import BeautifulSoup
import sys


PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import SPLITS

In [14]:
# Parent labels
PARENT_LABELS = [
    "decision",
    "legislation",
    "secondary sources",
    "unable to classify"
]

# Child/sub labels
SUBLABELS = [
    "authors",
    "title",
    "fragment",
    "citation",
    "source",
]

SPLITS = ["train", "dev", "test", "incoming", "silver"] 

In [15]:
per_split_totals = defaultdict(lambda: {
    "parent": 0,
    "child": 0,
    "total": 0,
})

for split in SPLITS:
    split_dir = PROJECT_ROOT / "data" / "annotated" / split

    if not split_dir.exists():
        print(f"skip (not found): {split_dir}")
        continue

    files = sorted(split_dir.rglob("*.html"))

    if not files:
        print(f"no html files in: {split_dir}")
        continue

    print(f"--- split: {split} (files: {len(files)}) ---")

    for f in files:
        try:
            text = f.read_text(encoding="utf-8", errors="ignore")
            soup = BeautifulSoup(text, "html.parser")

            # Find all manual_label and auto_label tags
            tags = soup.find_all(["manual_label", "auto_label"])

            parent_count = 0
            child_count = 0

            for tag in tags:
                label = tag.get("labelname")

                if not label:
                    continue

                if label in PARENT_LABELS:
                    parent_count += 1
                elif label in SUBLABELS:
                    child_count += 1

            total = parent_count + child_count

            per_split_totals[split]["parent"] += parent_count
            per_split_totals[split]["child"] += child_count
            per_split_totals[split]["total"] += total

            rel = f.relative_to(PROJECT_ROOT)
            print(
                f"{rel}: "
                f"parents={parent_count}, "
                f"children={child_count}, "
                f"total={total}"
            )

        except Exception as e:
            print(f"error processing {f}: {e}")

    print()

print("\n--- per-split totals ---")

for split in SPLITS:
    if split in per_split_totals:
        counts = per_split_totals[split]

        print(
            f"{split}: "
            f"parents={counts['parent']}, "
            f"children={counts['child']}, "
            f"total={counts['total']}"
        )

--- split: train (files: 4) ---
data\annotated\train\1994CanLII4528NLCA.html: parents=167, children=257, total=424
data\annotated\train\1997CanLII16226ONCA.html: parents=623, children=1288, total=1911
data\annotated\train\2008CSC9.html: parents=484, children=915, total=1399
data\annotated\train\2019SCC65.html: parents=1446, children=3212, total=4658

--- split: dev (files: 3) ---
data\annotated\dev\1989CanLII1415ONCA.html: parents=95, children=146, total=241
data\annotated\dev\2016NBOMB12.html: parents=136, children=170, total=306
data\annotated\dev\2021QCCA1675.html: parents=99, children=172, total=271

--- split: test (files: 4) ---
data\annotated\test\2001CanLII21117.html: parents=322, children=467, total=789
data\annotated\test\2002SCC33.html: parents=328, children=625, total=953
data\annotated\test\2005QCCA437.html: parents=117, children=195, total=312
data\annotated\test\2024NBKB203.html: parents=244, children=414, total=658

--- split: incoming (files: 34) ---
data\annotated\inc

In [16]:
# ---------------------------------------------------------
# Initialize counters
# ---------------------------------------------------------

per_split_totals = defaultdict(lambda: {
    "parent": 0,
    "child": 0,
    "total": 0,

    # Parent mention types
    "parent_types": defaultdict(int),

    # Child/component types
    "child_types": defaultdict(int),
})


# ---------------------------------------------------------
# Process files
# ---------------------------------------------------------

for split in SPLITS:
    split_dir = PROJECT_ROOT / "data" / "annotated" / split

    if not split_dir.exists():
        print(f"skip (not found): {split_dir}")
        continue

    files = sorted(split_dir.rglob("*.html"))

    if not files:
        print(f"no html files in: {split_dir}")
        continue

    print(f"--- split: {split} (files: {len(files)}) ---")

    for f in files:
        try:
            text = f.read_text(encoding="utf-8", errors="ignore")
            soup = BeautifulSoup(text, "html.parser")

            # Find all manual_label and auto_label tags
            tags = soup.find_all(["manual_label", "auto_label"])

            parent_count = 0
            child_count = 0

            for tag in tags:
                label = tag.get("labelname")

                if not label:
                    continue

                # -----------------------------
                # Parent label
                # -----------------------------
                if label in PARENT_LABELS:
                    parent_count += 1
                    per_split_totals[split]["parent_types"][label] += 1

                # -----------------------------
                # Child/component label
                # -----------------------------
                elif label in SUBLABELS:
                    child_count += 1
                    per_split_totals[split]["child_types"][label] += 1

            total = parent_count + child_count

            per_split_totals[split]["parent"] += parent_count
            per_split_totals[split]["child"] += child_count
            per_split_totals[split]["total"] += total

        except Exception as e:
            print(f"error processing {f}: {e}")

    print()


# ---------------------------------------------------------
# Print per-split summary
# ---------------------------------------------------------

print("\n" + "=" * 90)
print("PER-SPLIT LABEL BREAKDOWN")
print("=" * 90)

for split in SPLITS:
    if split not in per_split_totals:
        continue

    counts = per_split_totals[split]

    print(f"\n### {split}")
    print(f"Parents : {counts['parent']}")
    print(f"Children: {counts['child']}")
    print(f"Total   : {counts['total']}")

    print("\nParent mention types:")
    for label in PARENT_LABELS:
        print(f"  {label:<22} {counts['parent_types'][label]:>6}")

    print("\nComponent types:")
    for label in SUBLABELS:
        print(f"  {label:<22} {counts['child_types'][label]:>6}")

--- split: train (files: 4) ---



--- split: dev (files: 3) ---

--- split: test (files: 4) ---

--- split: incoming (files: 34) ---

--- split: silver (files: 55) ---


PER-SPLIT LABEL BREAKDOWN

### train
Parents : 2720
Children: 5672
Total   : 8392

Parent mention types:
  decision                 1580
  legislation               900
  secondary sources         237
  unable to classify          3

Component types:
  authors                   221
  title                    2065
  fragment                 1769
  citation                 1456
  source                    161

### dev
Parents : 330
Children: 488
Total   : 818

Parent mention types:
  decision                   96
  legislation               192
  secondary sources          42
  unable to classify          0

Component types:
  authors                    10
  title                     250
  fragment                  142
  citation                   71
  source                     15

### test
Parents : 1011
Children: 1701
Total   : 2712

Parent mention t

In [17]:
print("\n" + "=" * 140)
print("LABEL DISTRIBUTION BY SPLIT")
print("=" * 140)

header = (
    f"{'Split':<10}"
    f"{'Parents':>10}"
    f"{'Decision':>12}"
    f"{'Legislation':>14}"
    f"{'Secondary':>12}"
    f"{'Unclass.':>11}"
    f"{'Children':>11}"
    f"{'Authors':>10}"
    f"{'Title':>10}"
    f"{'Fragment':>11}"
    f"{'Citation':>11}"
    f"{'Source':>10}"
    f"{'Total':>10}"
)

print(header)
print("-" * 140)

for split in SPLITS:
    if split not in per_split_totals:
        continue

    c = per_split_totals[split]

    row = (
        f"{split:<10}"
        f"{c['parent']:>10}"
        f"{c['parent_types']['decision']:>12}"
        f"{c['parent_types']['legislation']:>14}"
        f"{c['parent_types']['secondary sources']:>12}"
        f"{c['parent_types']['unable to classify']:>11}"
        f"{c['child']:>11}"
        f"{c['child_types']['authors']:>10}"
        f"{c['child_types']['title']:>10}"
        f"{c['child_types']['fragment']:>11}"
        f"{c['child_types']['citation']:>11}"
        f"{c['child_types']['source']:>10}"
        f"{c['total']:>10}"
    )

    print(row)


LABEL DISTRIBUTION BY SPLIT
Split        Parents    Decision   Legislation   Secondary   Unclass.   Children   Authors     Title   Fragment   Citation    Source     Total
--------------------------------------------------------------------------------------------------------------------------------------------
train           2720        1580           900         237          3       5672       221      2065       1769       1456       161      8392
dev              330          96           192          42          0        488        10       250        142         71        15       818
test            1011         500           441          69          1       1701        49       677        647        290        38      2712
incoming        6995        3402          3119         387         87      12685       292      5584       3900       2677       232     19680
silver          6796        2522          3904         370          0      11339       169      5176       3811    

In [6]:
# Display names (edit freely)
PARENT_NAMES = {
    "decision":           "Decisions",
    "legislation":        "Legislation",
    "secondary sources":  "Sec. Sources",
    "unable to classify": "Unclassified",
}
SUB_NAMES = {
    "title":    "Title",
    "citation": "Citation",
    "fragment": "Fragment",
    "source":   "Source",
    "authors":  "Authors",
}
# Row display order
ROW_ORDER = ["title", "citation", "fragment", "source", "authors"]

PRINT_LATEX = True
TAGS = ["manual_label", "auto_label"]

# ---------------------------------------------------------
# Counters (aggregated over all splits)
# ---------------------------------------------------------
n_parents = defaultdict(int)                          # parent -> #mentions
n_with_comp = defaultdict(lambda: defaultdict(int))   # parent -> comp -> #mentions containing comp

for split in SPLITS:
    split_dir = PROJECT_ROOT / "data" / "annotated" / split
    if not split_dir.exists():
        print(f"skip (not found): {split_dir}")
        continue

    files = sorted(split_dir.rglob("*.html"))
    if not files:
        print(f"no html files in: {split_dir}")
        continue
    print(f"--- split: {split} (files: {len(files)}) ---")

    for f in files:
        try:
            soup = BeautifulSoup(f.read_text(encoding="utf-8", errors="ignore"), "html.parser")

            for parent in soup.find_all(TAGS):
                p_label = parent.get("labelname")
                if p_label not in PARENT_LABELS:
                    continue

                n_parents[p_label] += 1

                # distinct component types inside this mention
                present = {
                    c.get("labelname")
                    for c in parent.find_all(TAGS)
                    if c.get("labelname") in SUBLABELS
                }
                for comp in present:
                    n_with_comp[p_label][comp] += 1

        except Exception as e:
            print(f"error processing {f}: {e}")

# ---------------------------------------------------------
# Build table
# ---------------------------------------------------------
def coverage(p, c):
    if n_with_comp[p][c] == 0 or n_parents[p] == 0:
        return None
    return 100 * n_with_comp[p][c] / n_parents[p]

def fmt(v):
    return "---" if v is None else f"{v:.1f}%"

headers = ["Component"] + [PARENT_NAMES[p] for p in PARENT_LABELS]
rows = [[SUB_NAMES[c]] + [fmt(coverage(p, c)) for p in PARENT_LABELS] for c in ROW_ORDER]

# ---------------------------------------------------------
# Terminal print
# ---------------------------------------------------------
widths = [max(len(r[i]) for r in [headers] + rows) for i in range(len(headers))]

def line(cells):
    out = [cells[0].ljust(widths[0])] + [c.rjust(widths[i]) for i, c in enumerate(cells) if i > 0]
    return "| " + " | ".join(out) + " |"

sep = "+-" + "-+-".join("-" * w for w in widths) + "-+"

print("\n" + sep)
print(line(headers))
print(sep)
for r in rows:
    print(line(r))
print(sep)

n_str = "; ".join(f"n={n_parents[p]:,} {PARENT_NAMES[p].lower()}" for p in PARENT_LABELS)
print(f"Coverage of surface components in reference mentions by category ({n_str}).")

# ---------------------------------------------------------
# LaTeX output
# ---------------------------------------------------------
if PRINT_LATEX:
    def tex_cell(v):
        if v is None:
            return "---"
        s = f"{v:.1f}\\%"
        return "\\phantom{0}" + s if v < 10 else s

    def tex_header(h):
        return rf"\textbf{{{h.replace('Sec. ', 'Sec.\\ ')}}}"

    print("\n" + r"\begin{table}[h]")
    print(r"\centering")
    print(r"\small")
    print(r"\begin{tabular}{l" + "r" * len(PARENT_LABELS) + "}")
    print(r"\toprule")
    print(" & ".join(tex_header(h) for h in headers) + r" \\")
    print(r"\midrule")
    for c in ROW_ORDER:
        cells = [tex_cell(coverage(p, c)) for p in PARENT_LABELS]
        print(f"{SUB_NAMES[c]} & " + " & ".join(cells) + r" \\")
    print(r"\bottomrule")
    print(r"\end{tabular}")
    n_tex = "; ".join(
        f"$n={n_parents[p]:,}$ {PARENT_NAMES[p].lower()}".replace(",", "{,}")
        for p in PARENT_LABELS
    )
    print(rf"\caption{{Coverage of surface components in reference mentions by category ({n_tex}).}}")
    print(r"\label{tab:component-coverage}")
    print(r"\end{table}")

--- split: train (files: 4) ---
--- split: dev (files: 3) ---
--- split: extended_test (files: 38) ---
--- split: silver (files: 55) ---

+-----------+-----------+-------------+--------------+--------------+
| Component | Decisions | Legislation | Sec. Sources | Unclassified |
+-----------+-----------+-------------+--------------+--------------+
| Title     |     85.2% |       66.9% |        75.0% |          --- |
| Citation  |     46.4% |        9.9% |          --- |          --- |
| Fragment  |     37.7% |       64.3% |        48.8% |          --- |
| Source    |       --- |         --- |        54.7% |          --- |
| Authors   |       --- |         --- |        66.3% |          --- |
+-----------+-----------+-------------+--------------+--------------+
Coverage of surface components in reference mentions by category (n=8,100 decisions; n=8,556 legislation; n=1,105 sec. sources; n=91 unclassified).

\begin{table}[h]
\centering
\small
\begin{tabular}{lrrrr}
\toprule
\textbf{Componen

In [7]:


counts = Counter()

for split in SPLITS:

    split_dir = PROJECT_ROOT /"data" / "annotated" / split

    html_files = list(split_dir.rglob("*.html"))
    print(len(html_files), "html files found in", split_dir)
    for html_file in html_files:
        try:
            with open(html_file, "r", encoding="utf-8") as f:
                soup = BeautifulSoup(f, "html.parser")

            # Find all manual_label and auto_label tags
            tags = soup.find_all(["manual_label", "auto_label"])

            for tag in tags:
                label = tag.get("labelname")

                if label:
                    counts[label] += 1

        except Exception as e:
            print(f"Error processing {html_file}: {e}")

# =========================
# PRINT FINAL TABLE
# =========================

print("\n" + "=" * 80)
print("TOTAL LABEL COUNTS")
print("=" * 80)

print(f"{'LABEL':<20} {'COUNT':>10}")
print("-" * 32)

# Parent labels
parent_total = 0
for label in PARENT_LABELS:
    c = counts[label]
    parent_total += c
    print(f"{label:<20} {c:>10}")

print("-" * 32)
print(f"{'TOTAL PARENT':<20} {parent_total:>10}")

print()

# Sublabels
sub_total = 0
for label in SUBLABELS:
    c = counts[label]
    sub_total += c
    print(f"{label:<20} {c:>10}")

print("-" * 32)
print(f"{'TOTAL SUBLABELS':<20} {sub_total:>10}")

print()

# Grand total
grand_total = parent_total + sub_total
print(f"{'TOTAL COUNT':<20} {grand_total:>10}")

4 html files found in c:\Users\zakga\OneDrive\Documents\code\LeREaD\data\annotated\train
3 html files found in c:\Users\zakga\OneDrive\Documents\code\LeREaD\data\annotated\dev
38 html files found in c:\Users\zakga\OneDrive\Documents\code\LeREaD\data\annotated\extended_test
55 html files found in c:\Users\zakga\OneDrive\Documents\code\LeREaD\data\annotated\silver

TOTAL LABEL COUNTS
LABEL                     COUNT
--------------------------------
decision                   8100
legislation                8556
secondary sources          1105
unable to classify           91
--------------------------------
TOTAL PARENT              17852

authors                     741
title                     13752
fragment                  10269
citation                   6501
source                      622
--------------------------------
TOTAL SUBLABELS           31885

TOTAL COUNT               49737


### # Word tokens per document

In [8]:
per_split_totals = defaultdict(int)
per_split_word_counts = defaultdict(list)  # Store word counts per file

for split in SPLITS:
    split_dir = PROJECT_ROOT / "data" / "annotated" / split

    if not split_dir.exists():
        print(f"skip (not found): {split_dir}")
        continue

    files = sorted(split_dir.rglob("*.html"))

    if not files:
        print(f"no html files in: {split_dir}")
        continue

    print(f"--- split: {split} (files: {len(files)}) ---")

    for f in files:
        try:
            text = f.read_text(encoding="utf-8", errors="ignore")
            soup = BeautifulSoup(text, "html.parser")

            # Extract raw text from the entire HTML
            raw_text = soup.get_text(separator=" ", strip=True)
            word_tokens = len(raw_text.split())

            # Store word count for this file
            rel = f.relative_to(PROJECT_ROOT)
            per_split_word_counts[split].append((rel, word_tokens))

            # Print file and word count
            print(f"{rel}: {word_tokens} word tokens")

        except Exception as e:
            print(f"error processing {f}: {e}")

    print()

# Print fancy summary
print("\n" + "="*60)
print("         WORD TOKEN COUNT SUMMARY         ")
print("="*60)
for split in SPLITS:
    if split in per_split_word_counts:
        print(f"\n--- {split.upper()} ---")
        for rel, count in per_split_word_counts[split]:
            print(f"  {rel}: {count}")
        total = sum(count for _, count in per_split_word_counts[split])
        print(f"  Total for {split}: {total} word tokens")
print("="*60)

--- split: train (files: 4) ---
data\annotated\train\1994CanLII4528NLCA.html: 10144 word tokens
data\annotated\train\1997CanLII16226ONCA.html: 40540 word tokens
data\annotated\train\2008CSC9.html: 28442 word tokens
data\annotated\train\2019SCC65.html: 65440 word tokens

--- split: dev (files: 3) ---
data\annotated\dev\1989CanLII1415ONCA.html: 4385 word tokens
data\annotated\dev\2016NBOMB12.html: 7671 word tokens
data\annotated\dev\2021QCCA1675.html: 4773 word tokens

--- split: extended_test (files: 38) ---
data\annotated\extended_test\1983CanLII2967L.html: 2505 word tokens
data\annotated\extended_test\1993CanLII1889PESCTSD.html: 10377 word tokens
data\annotated\extended_test\1993CanLII3004FCA.html: 22255 word tokens
data\annotated\extended_test\1993NSSC71.html: 5996 word tokens
data\annotated\extended_test\1997CanLII11497.html: 5250 word tokens
data\annotated\extended_test\1999CanLII7320(ONCA).html: 23320 word tokens
data\annotated\extended_test\2001BCSC1342.html: 5383 word tokens
dat